# Dekomposisi QR

Notebook ini membahas konsep **Dekomposisi QR**, salah satu metode pemecahan matriks yang sangat penting dalam komputasi aljabar linier numerik. Kita akan mempelajari dasar teorinya, mengimplementasikan metode **Gram-Schmidt** secara manual, dan membandingkannya dengan fungsi bawaan **NumPy**.

## 1. Apa itu Dekomposisi QR?

Dekomposisi QR adalah pemecahan matriks riil berukuran $m \times n$ ($A$) menjadi hasil kali dua matriks:
$$A = QR$$

Di mana:
- $Q$ adalah matriks berukuran $m \times n$ yang memiliki kolom-kolom **ortonormal** (sehingga $Q^T Q = I$).
- $R$ adalah matriks segitiga atas (upper triangular) berukuran $n \times n$.

### Kegunaan:
- Menyelesaikan Sistem Persamaan Linear (lebih stabil secara numerik dibanding invers matriks biasa).
- Menghitung masalah kuadrat terkecil (*least squares regression*).
- Algoritma pencarian nilai eigen (*eigenvalue algorithms*).

## 2. Ortogonalisasi Gram-Schmidt

Metode Gram-Schmidt digunakan untuk mengubah sekumpulan vektor yang bebas linear menjadi sekumpulan vektor yang ortonormal.

Diberikan kolom-kolom matriks $A$ sebagai vektor $a_1, a_2, \dots, a_n$, kita mencari vektor ortogonal $u_1, u_2, \dots, u_n$:
$$
\begin{aligned}
u_1 &= a_1 \\
u_2 &= a_2 - \text{proj}_{u_1}(a_2) \\
u_3 &= a_3 - \text{proj}_{u_1}(a_3) - \text{proj}_{u_2}(a_3) \\
&\ \ \vdots \\
u_k &= a_k - \sum_{j=1}^{k-1} \text{proj}_{u_j}(a_k)
\end{aligned}
$$

Di mana proyeksi didefinisikan sebagai:
$$\text{proj}_{u}(a) = \frac{\langle a, u \rangle}{\langle u, u \rangle} u$$

Setelah mendapatkan vektor-vektor ortogonal $u_i$, kita menormalisasinya menjadi vektor satuan ortonormal $q_i = \frac{u_i}{\|u_i\|}$ untuk membentuk matriks $Q$. Matriks $R$ kemudian dapat diperoleh dengan $R = Q^T A$.

In [ ]:
import numpy as np

def gram_schmidt_qr(A):
    A = A.astype(float)
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    
    for j in range(n):
        v = A[:, j]
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v = v - R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        if R[j, j] == 0:
            raise ValueError("Kolom-kolom matriks tidak bebas linier.")
        Q[:, j] = v / R[j, j]
        
    return Q, R

# Definisikan matriks A (3x3)
A = np.array([[12.0, -51.0, 4.0],
              [6.0, 167.0, -68.0],
              [-4.0, 24.0, -41.0]])

print("Matriks Asal A:\n", A)

# Hitung Q dan R
Q_gs, R_gs = gram_schmidt_qr(A)
print("\nMatriks Q (Ortogonal - hasil Gram-Schmidt):\n", Q_gs)
print("\nMatriks R (Segitiga Atas - hasil Gram-Schmidt):\n", R_gs)

## 3. Menggunakan NumPy `np.linalg.qr`

NumPy memiliki fungsi internal `np.linalg.qr` yang diimplementasikan menggunakan Householder Reflections (lebih stabil secara numerik untuk matriks besar).

In [ ]:
# Hitung QR menggunakan numpy
Q_np, R_np = np.linalg.qr(A)

print("Matriks Q (NumPy):\n", Q_np)
print("\nMatriks R (NumPy):\n", R_np)

# Verifikasi perkalian Q x R
A_reconstructed = np.dot(Q_np, R_np)
print("\nHasil rekonstruksi A (Q x R):\n", A_reconstructed)
print("\nApakah rekonstruksi sama dengan matriks asli?", np.allclose(A, A_reconstructed))